## Preprocess the data

In [ ]:
import numpy as np 
import pandas as pd

df = pd.read_csv('../EDA/train_dataset_final.csv')

In [2]:
def convert_categorize(df):
    import pandas as pd

    # Define category orders
    sleep_categories = ['< 5 hours', '5-6 hours', '7-8 hours', '> 8 hours']
    habit_categories = ['Unhealthy', 'Moderate', 'Healthy']
    age_categories = ['<20', '20-30', '30-40', '40-50', '> 50']
    age_risk = ['< 30','30+']
    risk_level = ['Low Risk','High Risk']
    financial_order = [1,2,3,4,5]
    work_study_order = [0,1,2,3,4,5]
    work_hour = [i for i in range(13)]

    # Convert columns to ordered categorical types if they exist
    df['Sleep Duration'] = pd.Categorical(df['Sleep Duration'], categories=sleep_categories, ordered=True)
    df['Dietary Habits'] = pd.Categorical(df['Dietary Habits'], categories=habit_categories, ordered=True)
    df['Age_Group'] = df['Age_Group'].str.strip()
    df['Age_Group'] = pd.Categorical(df['Age_Group'], categories=age_categories, ordered=True)
    df['Age_Risk'] = pd.Categorical(df['Age_Risk'],categories=age_risk, ordered=True)
    df['Risk_Level'] = pd.Categorical(df['Risk_Level'], categories=risk_level, ordered=True)
    df['Academic Pressure'] = pd.Categorical(df['Academic Pressure'],categories=work_study_order,ordered=True)
    df['Work Pressure'] = pd.Categorical(df['Work Pressure'], categories=work_study_order,ordered=True)
    df['Study Satisfaction'] = pd.Categorical(df['Study Satisfaction'],categories=work_study_order, ordered=True)
    df['Job Satisfaction'] = pd.Categorical(df['Job Satisfaction'], categories=work_study_order,ordered=True)
    df['Financial Stress'] = pd.Categorical(df['Financial Stress'],ordered=True,categories=financial_order)
    df['Work/Study Hours'] = pd.Categorical(df['Work/Study Hours'], ordered=True, categories=work_hour)
    
    ### Change to category types:
    object_cols = df.select_dtypes(include="object").columns
    for col in object_cols:
        df[col] = df[col].astype("category")
    
    return df
df = convert_categorize(df)

In [3]:
neg = np.sum(df['Depression'] == 0)
pos = np.sum(df['Depression'] == 1)

print(f"Negatives (0): {neg}, Positives (1): {pos}")
scale_pos_weight = neg / pos
print("scale_pos_weight =", round(scale_pos_weight,2))

Negatives (0): 115133, Positives (1): 25567
scale_pos_weight = 4.5


## Spliting model:

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix


X = df.drop('Depression',axis=1)
y = df['Depression']
X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y,random_state=42)
X_train.columns = [c.replace(" ", "_") for c in X_train.columns]
X_val.columns = [c.replace(" ", "_") for c in X_val.columns]

## Optuna XGBoost

In [5]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import optuna
import xgboost as xgb
from sklearn.metrics import roc_auc_score

# ----------------------------
# Prepare DMatrix
# ----------------------------
dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
dval = xgb.DMatrix(X_val, label=y_val, enable_categorical=True)

# ----------------------------
# Define objective
# ----------------------------
def objective(trial):
    param = {
        'objective': 'binary:logistic',
        'tree_method': 'hist',
        'booster': trial.suggest_categorical('booster', ['gbtree', 'dart']),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'eta': trial.suggest_float('eta', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'alpha': trial.suggest_float('alpha', 0, 5),
        'lambda': trial.suggest_float('lambda', 0, 5),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'scale_pos_weight': scale_pos_weight,
        'enable_categorical': True,
        'n_jobs': -1,
        'eval_metric': 'auc'
    }

    # Tune num_boost_round (max rounds) with Optuna
    num_boost_round = trial.suggest_int('num_boost_round', 100, 2000)

    bst = xgb.train(
        params=param,
        dtrain=dtrain,
        num_boost_round=num_boost_round,
        evals=[(dval, 'validation')],
        early_stopping_rounds=15,
        verbose_eval=False
    )

    y_pred = bst.predict(dval)
    auc = roc_auc_score(y_val, y_pred)
    return auc

# ----------------------------
# Run Optuna study with 30 trials
# ----------------------------
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

# ----------------------------+
# Results
# ----------------------------
print("Best ROC-AUC:", study.best_value)
print("Best params:", study.best_params)
# Best ROC-AUC: 0.9741500526989383
# Best params: {'booster': 'gbtree', 'max_depth': 4, 'eta': 0.0411902266055743, 'subsample': 0.7976775594616186, 'colsample_bytree': 0.6305450751517756, 'gamma': 1.2833573484902328, 'alpha': 2.7742682567338655, 'lambda': 0.005956096352296214, 'min_child_weight': 6, 'num_boost_round': 822}

# Best ROC-AUC: 0.9742347924569752
# Best params: {'booster': 'gbtree', 'max_depth': 4, 'eta': 0.020137217178148186, 'subsample': 0.8183990740458535, 'colsample_bytree': 0.643872724219229, 'gamma': 1.1831197843411665, 'alpha': 2.054415451246265, 'lambda': 2.9550125041513784, 'min_child_weight': 7, 'num_boost_round': 977}

# Best ROC-AUC: 0.9740182428466394
# Best params: {'boosting_type': 'dart', 'num_leaves': 213, 'max_depth': 3, 'learning_rate': 0.05025460436017625, 'min_child_samples': 80, 'subsample': 0.753636168367769, 'colsample_bytree': 0.9095444750066676, 'reg_alpha': 1.6209308403690994, 'reg_lambda': 2.3977397413994055, 'num_boost_round': 1426}

[I 2025-11-17 01:03:17,683] A new study created in memory with name: no-name-b1b1672e-cba8-4ed1-96ed-a7c1b622f62e
[I 2025-11-17 01:03:23,485] Trial 0 finished with value: 0.9739057991312752 and parameters: {'booster': 'gbtree', 'max_depth': 10, 'eta': 0.011306035110239235, 'subsample': 0.6840741093431115, 'colsample_bytree': 0.7465525079550489, 'gamma': 1.7616333288505621, 'alpha': 3.3870736151587515, 'lambda': 3.9694591087994198, 'min_child_weight': 6, 'num_boost_round': 1288}. Best is trial 0 with value: 0.9739057991312752.
[I 2025-11-17 01:03:34,638] Trial 1 finished with value: 0.9729825214064571 and parameters: {'booster': 'dart', 'max_depth': 9, 'eta': 0.09348736643500068, 'subsample': 0.8055172674443525, 'colsample_bytree': 0.8000294875044235, 'gamma': 1.7190513604279922, 'alpha': 3.826486598417268, 'lambda': 0.5590735309495287, 'min_child_weight': 6, 'num_boost_round': 1267}. Best is trial 0 with value: 0.9739057991312752.
[I 2025-11-17 01:05:16,743] Trial 2 finished with value

Best ROC-AUC: 0.9742347924569752
Best params: {'booster': 'gbtree', 'max_depth': 4, 'eta': 0.020137217178148186, 'subsample': 0.8183990740458535, 'colsample_bytree': 0.643872724219229, 'gamma': 1.1831197843411665, 'alpha': 2.054415451246265, 'lambda': 2.9550125041513784, 'min_child_weight': 7, 'num_boost_round': 977}


## Optuna LightGBM

In [6]:
# import warnings
# warnings.filterwarnings("ignore")

import optuna
from lightgbm import LGBMClassifier, early_stopping
from sklearn.metrics import roc_auc_score

categorical_cols = X_train.select_dtypes('category').columns.tolist()

# ----------------------------
# Define objective for Optuna
# ----------------------------
def objective(trial):
    model = LGBMClassifier(
        objective='binary',
        boosting_type=trial.suggest_categorical('boosting_type', ['gbdt', 'dart']),
        num_leaves=trial.suggest_int('num_leaves', 16, 256),
        max_depth=trial.suggest_int('max_depth', 3, 12),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        min_child_samples=trial.suggest_int('min_child_samples', 5, 100),
        subsample=trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.6, 1.0),
        reg_alpha=trial.suggest_float('reg_alpha', 0.0, 5.0),
        reg_lambda=trial.suggest_float('reg_lambda', 0.0, 5.0),
        scale_pos_weight=4.5,
        n_estimators=trial.suggest_int('num_boost_round', 100, 2000),
        n_jobs=-1
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='auc',
        # early_stopping_rounds=15,
        categorical_feature=categorical_cols,
        # verbose=False
    )

    y_pred = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, y_pred)
    return auc

# ----------------------------
# Run Optuna study
# ----------------------------
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

# ----------------------------
# Results
# ----------------------------
print("Best ROC-AUC:", study.best_value)
print("Best params:", study.best_params)
# Best ROC-AUC: 0.9742082299222566
# Best params: {'boosting_type': 'gbdt', 'num_leaves': 244, 'max_depth': 3, 'learning_rate': 0.014956339718402342, 'min_child_samples': 37, 'subsample': 0.8301813189670537, 'colsample_bytree': 0.6908968234541976, 'reg_alpha': 3.3524586081601617, 'reg_lambda': 2.0787174907751496, 'num_boost_round': 1260}

[I 2025-11-17 01:17:12,177] A new study created in memory with name: no-name-96fe2a3f-6b37-426d-81b8-d7024e258ce8


[LightGBM] [Info] Number of positive: 19175, number of negative: 86350
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004533 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 461
[LightGBM] [Info] Number of data points in the train set: 105525, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.181710 -> initscore=-1.504801
[LightGBM] [Info] Start training from score -1.504801
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

[I 2025-11-17 01:17:26,694] Trial 0 finished with value: 0.9736875564107359 and parameters: {'boosting_type': 'dart', 'num_leaves': 231, 'max_depth': 5, 'learning_rate': 0.0746312984951037, 'min_child_samples': 11, 'subsample': 0.8051102703550244, 'colsample_bytree': 0.8467204048487263, 'reg_alpha': 1.929646341690478, 'reg_lambda': 4.678686789160858, 'num_boost_round': 450}. Best is trial 0 with value: 0.9736875564107359.


[LightGBM] [Info] Number of positive: 19175, number of negative: 86350
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002860 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 461
[LightGBM] [Info] Number of data points in the train set: 105525, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.181710 -> initscore=-1.504801
[LightGBM] [Info] Start training from score -1.504801
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

[I 2025-11-17 01:17:37,247] Trial 1 finished with value: 0.9687892391198618 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 190, 'max_depth': 8, 'learning_rate': 0.20423804360830253, 'min_child_samples': 41, 'subsample': 0.9982859998415996, 'colsample_bytree': 0.8170948922647215, 'reg_alpha': 4.8480244144826425, 'reg_lambda': 4.076009168176054, 'num_boost_round': 1909}. Best is trial 0 with value: 0.9736875564107359.


...

[LightGBM] [Info] Number of positive: 19175, number of negative: 86350
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004121 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 461
[LightGBM] [Info] Number of data points in the train set: 105525, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.181710 -> initscore=-1.504801
[LightGBM] [Info] Start training from score -1.504801
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

[I 2025-11-17 07:09:10,171] Trial 28 finished with value: 0.9740182428466394 and parameters: {'boosting_type': 'dart', 'num_leaves': 213, 'max_depth': 3, 'learning_rate': 0.05025460436017625, 'min_child_samples': 80, 'subsample': 0.753636168367769, 'colsample_bytree': 0.9095444750066676, 'reg_alpha': 1.6209308403690994, 'reg_lambda': 2.3977397413994055, 'num_boost_round': 1426}. Best is trial 28 with value: 0.9740182428466394.


[LightGBM] [Info] Number of positive: 19175, number of negative: 86350
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002250 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 461
[LightGBM] [Info] Number of data points in the train set: 105525, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.181710 -> initscore=-1.504801
[LightGBM] [Info] Start training from score -1.504801
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

[I 2025-11-17 07:11:42,933] Trial 29 finished with value: 0.9710910265180952 and parameters: {'boosting_type': 'dart', 'num_leaves': 236, 'max_depth': 10, 'learning_rate': 0.06961884908889264, 'min_child_samples': 81, 'subsample': 0.6658955464516616, 'colsample_bytree': 0.9974772942828378, 'reg_alpha': 2.3135859402663552, 'reg_lambda': 1.9860234398564647, 'num_boost_round': 1355}. Best is trial 28 with value: 0.9740182428466394.


Best ROC-AUC: 0.9740182428466394
Best params: {'boosting_type': 'dart', 'num_leaves': 213, 'max_depth': 3, 'learning_rate': 0.05025460436017625, 'min_child_samples': 80, 'subsample': 0.753636168367769, 'colsample_bytree': 0.9095444750066676, 'reg_alpha': 1.6209308403690994, 'reg_lambda': 2.3977397413994055, 'num_boost_round': 1426}


## Optuna Catboost:

In [7]:
import optuna
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 2000),  # num_boost_round
        'depth': trial.suggest_int('depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'scale_pos_weight': scale_pos_weight,
        'task_type': 'CPU',
        'random_seed': 42,
        'logging_level': 'Silent'
    }

    model = CatBoostClassifier(**params)
    
    train_pool = Pool(X_train, y_train, cat_features=categorical_cols)
    val_pool = Pool(X_val, y_val, cat_features=categorical_cols)
    
    model.fit(
        train_pool,
        eval_set=val_pool,
        early_stopping_rounds=15,
        verbose=False
    )
    
    y_pred = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, y_pred)
    return auc  # Optuna maximizes by default

# ----------------------------
# Run Optuna study
# ----------------------------
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

# ----------------------------
# Best results
# ----------------------------
print("Best ROC-AUC:", study.best_value)
print("Best params:", study.best_params)

# ----------------------------
# Train final model
# ----------------------------
best_params = study.best_params
best_params.update({
    'scale_pos_weight': scale_pos_weight,
    'task_type': 'CPU',
    'random_seed': 42,
    'logging_level': 'Silent'
})

final_model = CatBoostClassifier(**best_params)
final_model.fit(Pool(X_train, y_train, cat_features=categorical_cols))
y_pred_final = final_model.predict_proba(Pool(X_val, y_val, cat_features=categorical_cols))[:, 1]
final_auc = roc_auc_score(y_val, y_pred_final)
print(f"Final Validation ROC-AUC: {final_auc:.6f}")
# Best ROC-AUC: 0.9745803065161056
# Best params: {'iterations': 1443, 'depth': 6, 'learning_rate': 0.04798820473738284, 'l2_leaf_reg': 4.745943265136245, 'bagging_temperature': 0.6651805464938891, 'border_count': 205}
# Final Validation ROC-AUC: 0.974607

Best ROC-AUC: 0.9746379809699414
Best params: {'iterations': 593, 'depth': 5, 'learning_rate': 0.06156867848924621, 'l2_leaf_reg': 2.8722545484680766, 'bagging_temperature': 0.48087662422549526, 'border_count': 93}
Final Validation ROC-AUC: 0.974650

[I 2025-11-17 07:11:43,061] A new study created in memory with name: no-name-3af7b271-bb82-4c16-8c40-98462a6c4e89
[I 2025-11-17 07:12:34,799] Trial 0 finished with value: 0.9740055513142948 and parameters: {'iterations': 1692, 'depth': 3, 'learning_rate': 0.010960205836472506, 'l2_leaf_reg': 7.9727402062152475, 'bagging_temperature': 0.9815284736071216, 'border_count': 159}. Best is trial 0 with value: 0.9740055513142948.
[I 2025-11-17 07:12:58,027] Trial 1 finished with value: 0.9745859103575818 and parameters: {'iterations': 1628, 'depth': 5, 'learning_rate': 0.07177141750377972, 'l2_leaf_reg': 3.049921016269179, 'bagging_temperature': 0.5467222127735817, 'border_count': 173}. Best is trial 1 with value: 0.9745859103575818.
[I 2025-11-17 07:14:54,629] Trial 2 finished with value: 0.9741833632154149 and parameters: {'iterations': 908, 'depth': 10, 'learning_rate': 0.010239050657217579, 'l2_leaf_reg': 6.763262978722057, 'bagging_temperature': 0.7134375881906896, 'border_count': 211}. B

Best ROC-AUC: 0.9746379809699414
Best params: {'iterations': 593, 'depth': 5, 'learning_rate': 0.06156867848924621, 'l2_leaf_reg': 2.8722545484680766, 'bagging_temperature': 0.48087662422549526, 'border_count': 93}
Final Validation ROC-AUC: 0.974650
